<a href="https://colab.research.google.com/github/rameen57/nlp-assignment-1/blob/main/NLP_Assignment_1_Q5_Q6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Question 5: Audit and clean a campus SMS sample

### Task 1: Data audit (missing, duplicates, counts)

In [10]:
import pandas as pd, unicodedata, re

# Step 1: read CSV as UTF-8, keep the four columns
raw = pd.read_csv("AIDC3063_Assignment_01_Data.csv", encoding="utf-8", keep_default_na=True)[["id","split","label","message"]]

# Step 2: drop missing / whitespace-only messages
blank_mask = raw["message"].fillna("").str.strip() == ""
missing_ids = raw.loc[blank_mask, "id"].tolist()
pre = raw.loc[~blank_mask].copy()

# Steps 3-4: keep raw, create NFC
pre["raw_message"] = pre["message"]
pre["nfc_message"] = pre["raw_message"].map(lambda s: unicodedata.normalize("NFC", s))

# Step 5: dedupe within split on (label, nfc_message), keep first
df = pre.drop_duplicates(subset=["split","label","nfc_message"], keep="first").copy()
dup_ids = sorted(set(pre["id"]) - set(df["id"]))

print("Missing-message IDs removed:", missing_ids)
print("Post-NFC duplicate IDs removed:", dup_ids)
print("\nFinal rows by split:\n", df.groupby("split").size())
print("\nFinal class counts by split:\n", df.groupby(["split","label"]).size())

# Cross-split check
shared = set(df[df.split=="train"]["nfc_message"]) & set(df[df.split=="validation"]["nfc_message"])
print("\nNFC-equal messages shared across splits:", shared if shared else "None")

Missing-message IDs removed: ['T10']
Post-NFC duplicate IDs removed: ['T09', 'T11']

Final rows by split:
 split
train         9
validation    4
dtype: int64

Final class counts by split:
 split       label
train       ham      5
            spam     4
validation  ham      2
            spam     2
dtype: int64

NFC-equal messages shared across splits: None


### Task 2: Pipeline A (steps 6 to 9) and retained records

In [11]:
stop = {"a","at","is","the","your"}

def clean(s):
    s = s.lower()                        # (6) lowercase
    s = re.sub(r"[^\w\s]", " ", s)       # (7) punctuation -> space (Unicode-aware)
    tokens = s.split()                   # (8) split on whitespace
    tokens = [t for t in tokens if t not in stop]
    return " ".join(tokens)              # (9) rejoin

df["processed_message"] = df["nfc_message"].map(clean).fillna("")
df[["id","label","raw_message","processed_message"]].to_string(index=False)
print(df[["id","label","raw_message","processed_message"]].to_string(index=False))

 id label                    raw_message           processed_message
T01   ham    Your library book is ready.          library book ready
T02  spam  FREE campus prize! Claim now. free campus prize claim now
T03   ham   The seminar starts at 10:00.        seminar starts 10 00
T04  spam WIN a free voucher. Click now!  win free voucher click now
T05   ham     Your fee receipt is ready.           fee receipt ready
T06  spam   Claim your FREE voucher now.      claim free voucher now
T07   ham    The café meetup is at noon.            café meetup noon
T08  spam         Urgent! Claim a prize.          urgent claim prize
T12   ham        This event is not free.         this event not free
V01   ham   The workshop starts at noon.        workshop starts noon
V02  spam           Win a voucher today.           win voucher today
V03   ham          Your book is overdue.                book overdue
V04  spam          Claim your prize now.             claim prize now


### Task 3: T12 stages, and NFC on T07 and T11

In [12]:
s = pre.loc[pre.id=="T12","nfc_message"].iloc[0]
a = s.lower();                      print("1. lowercase      :", a)
b = re.sub(r"[^\w\s]", " ", a);     print("2. punctuation    :", repr(b))
c = b.split();                      print("3. tokenization   :", c)
d = [t for t in c if t not in stop];print("4. stop-word removal:", d)

print()
for i in ["T07","T11"]:
    r = pre.loc[pre.id==i].iloc[0]
    print(i, "raw:", ascii(r.raw_message), "| NFC:", ascii(r.nfc_message))
print("T07 == T11 after NFC:", pre.loc[pre.id=="T07","nfc_message"].iloc[0] == pre.loc[pre.id=="T11","nfc_message"].iloc[0])

1. lowercase      : this event is not free.
2. punctuation    : 'this event is not free '
3. tokenization   : ['this', 'event', 'is', 'not', 'free']
4. stop-word removal: ['this', 'event', 'not', 'free']

T07 raw: 'The caf\xe9 meetup is at noon.' | NFC: 'The caf\xe9 meetup is at noon.'
T11 raw: 'The cafe\u0301 meetup is at noon.' | NFC: 'The caf\xe9 meetup is at noon.'
T07 == T11 after NFC: True


### Task 4: Pipeline B (Porter stemming after Pipeline A stop-word filtering)
Rule: take Pipeline A's `processed_message` and stem each token with NLTK `PorterStemmer` (default NLTK_EXTENSIONS mode). Pipeline A is not replaced.

In [13]:
# !pip install nltk   # uncomment if needed
from nltk.stem import PorterStemmer
ps = PorterStemmer()
df["processed_B"] = df["processed_message"].map(lambda s: " ".join(ps.stem(t) for t in s.split()))
print(df.loc[df.id.isin(["T01","T12"]), ["id","processed_message","processed_B"]].to_string(index=False))

 id   processed_message        processed_B
T01  library book ready librari book readi
T12 this event not free thi event not free


## Question 6: From cleaned SMS to numerical evidence
Uses Pipeline A `processed_message` from Question 5 (not Pipeline B).

### Task 1: Count and TF-IDF matrices (fit on train only)

In [14]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

kw = dict(analyzer="word", tokenizer=str.split, token_pattern=None, preprocessor=None,
          lowercase=False, ngram_range=(1,1), min_df=1, max_df=1.0,
          max_features=None, strip_accents=None, stop_words=None)

tr = df[df.split=="train"].reset_index(drop=True)
va = df[df.split=="validation"].reset_index(drop=True)

cv = CountVectorizer(binary=False, **kw)
Xtr = cv.fit_transform(tr.processed_message)      # fit on TRAIN only
Xva = cv.transform(va.processed_message)          # transform only

tv = TfidfVectorizer(use_idf=True, smooth_idf=False, sublinear_tf=False, norm=None, binary=False, **kw)
Ttr = tv.fit_transform(tr.processed_message)
Tva = tv.transform(va.processed_message)

vocab = list(cv.get_feature_names_out())
print("Train count shape      :", Xtr.shape)
print("Validation count shape :", Xva.shape)
print("Train TF-IDF shape     :", Ttr.shape)
print("Validation TF-IDF shape:", Tva.shape)
print("Training vocabulary size:", len(vocab))
print(vocab)

Train count shape      : (9, 24)
Validation count shape : (4, 24)
Train TF-IDF shape     : (9, 24)
Validation TF-IDF shape: (4, 24)
Training vocabulary size: 24
['00', '10', 'book', 'café', 'campus', 'claim', 'click', 'event', 'fee', 'free', 'library', 'meetup', 'noon', 'not', 'now', 'prize', 'ready', 'receipt', 'seminar', 'starts', 'this', 'urgent', 'voucher', 'win']


In [15]:
# Labeled count submatrix for T04, T06, V03
terms = ["claim","free","now","voucher","overdue"]
def count_row(doc_id):
    if doc_id in set(tr.id):
        row = Xtr[tr.index[tr.id==doc_id][0]].toarray()[0]
    else:
        row = Xva[va.index[va.id==doc_id][0]].toarray()[0]
    return {t: (int(row[vocab.index(t)]) if t in vocab else "absent") for t in terms}

sub = pd.DataFrame({i: count_row(i) for i in ["T04","T06","V03"]}).T
print(sub)
print("\n'overdue' is not in the training vocabulary, so it has no column (absent).")

    claim free now voucher overdue
T04     0    1   1       1  absent
T06     1    1   1       1  absent
V03     0    0   0       0  absent

'overdue' is not in the training vocabulary, so it has no column (absent).


### Task 2: Manual TF-IDF for `voucher` in T04

In [16]:
N = Xtr.shape[0]
col = vocab.index("voucher")
count = int(Xtr[tr.index[tr.id=="T04"][0], col])
df_t = int((Xtr[:, col].toarray() > 0).sum())
idf = np.log(N/df_t) + 1
manual = count * idf
vec = Ttr[tr.index[tr.id=="T04"][0], col]

print(f"count(voucher, T04) = {count}")
print(f"N (training documents) = {N}")
print(f"df(voucher) = {df_t}")
print(f"IDF = ln({N}/{df_t}) + 1 = {idf:.6f}")
print(f"manual TF-IDF = {count} x {idf:.6f} = {manual:.6f}")
print(f"vectorizer cell = {vec!r}")
print("Match within 1e-6:", abs(manual - vec) < 1e-6)

count(voucher, T04) = 1
N (training documents) = 9
df(voucher) = 2
IDF = ln(9/2) + 1 = 2.504077
manual TF-IDF = 1 x 2.504077 = 2.504077
vectorizer cell = np.float64(2.504077396776274)
Match within 1e-6: True


### Task 3: Cosine similarity of T04 and T06 (full count vectors)

In [17]:
a = Xtr[tr.index[tr.id=="T04"][0]].toarray()[0]
b = Xtr[tr.index[tr.id=="T06"][0]].toarray()[0]

overlap = [vocab[i] for i in range(len(vocab)) if a[i]>0 and b[i]>0]
dot = a @ b
na, nb_ = np.linalg.norm(a), np.linalg.norm(b)
print("Non-zero overlap:", overlap)
print("Dot product     :", dot)
print(f"||T04|| = sqrt({int((a**2).sum())}) = {na:.4f}")
print(f"||T06|| = sqrt({int((b**2).sum())}) = {nb_:.4f}")
print(f"cosine = {dot} / ({na:.4f} x {nb_:.4f}) = {dot/(na*nb_):.4f}")

Non-zero overlap: ['free', 'now', 'voucher']
Dot product     : 3
||T04|| = sqrt(5) = 2.2361
||T06|| = sqrt(4) = 2.0000
cosine = 3 / (2.2361 x 2.0000) = 0.6708
